# Dashboard V2 Analytics Playground

This notebook is a lightweight starting point for ad hoc analysis against the processed Parquet outputs.

It is designed to work without local raw CSV files as long as these files exist:

- `data/processed/consumer_master.parquet`
- `data/processed/consumption.parquet`
- `data/processed/vend.parquet`

The notebook also reuses the project analytics helpers so the exploratory outputs stay aligned with the Streamlit dashboard logic.

In [ ]:
from __future__ import annotations

# Standard library imports for path resolution and repo-local imports.
from pathlib import Path
import sys

# Core analysis libraries used throughout the notebook.
import pandas as pd
import plotly.express as px
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    # Walk upward until we find the repository layout this notebook expects.
    start_path = (start or Path.cwd()).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "src").exists() and (candidate / "config").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing src/ and config/.")


# Add the repo root to sys.path so local project modules can be imported.
PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from src.analytics import build_dashboard_analytics
from src.path_utils import load_app_config


# Load the same app config the Streamlit dashboard uses.
APP_CONFIG = load_app_config(PROJECT_ROOT / "config" / "app_config.yaml")
# Point to the processed-data folder so this notebook works without raw CSVs.
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

consumer_master_path = PROCESSED_DIR / "consumer_master.parquet"
consumption_path = PROCESSED_DIR / "consumption.parquet"
vend_path = PROCESSED_DIR / "vend.parquet"

# Quick existence check before trying to load the Parquet files.
for path in [consumer_master_path, consumption_path, vend_path]:
    print(path.name, "exists" if path.exists() else "missing", path)

# Load the processed datasets exactly as persisted by the dashboard pipeline.
consumer_master_df = pd.read_parquet(consumer_master_path)
consumption_df = pd.read_parquet(consumption_path)
vend_df = pd.read_parquet(vend_path)

# Rebuild the derived analytics layer so notebook outputs stay aligned with dashboard logic.
analytics = build_dashboard_analytics(
    master_df=consumer_master_df,
    vend_df=vend_df,
    consumption_df=consumption_df,
    app_config=APP_CONFIG,
)

# Pull out the most useful derived tables into short variable names for exploration.
consumer_summary = analytics["datasets"]["consumer_summary"]
vend_enriched = analytics["datasets"]["vend_enriched"]
consumption_enriched = analytics["datasets"]["consumption_enriched"]
exception_summary = analytics["exception_summary"]
portfolio_metrics = analytics["portfolio_metrics"]
vend_timestamp_quality = analytics["vend_timestamp_quality"]

In [ ]:
# High-level shape check for both the base and derived datasets.
dataset_overview = pd.DataFrame(
    [
        {"dataset": "consumer_master", "rows": len(consumer_master_df), "columns": len(consumer_master_df.columns)},
        {"dataset": "consumption", "rows": len(consumption_df), "columns": len(consumption_df.columns)},
        {"dataset": "vend", "rows": len(vend_df), "columns": len(vend_df.columns)},
        {"dataset": "consumer_summary", "rows": len(consumer_summary), "columns": len(consumer_summary.columns)},
    ]
)

# Portfolio metrics and vend timestamp quality are useful early sanity checks.
display(dataset_overview)
display(pd.DataFrame([portfolio_metrics]))
display(pd.DataFrame([vend_timestamp_quality]))

## Quick checks

A few high-signal tables to sanity-check the processed outputs and the derived analytics layer.

In [ ]:
# Exception summary gives the quickest view of operational hotspots.
display(exception_summary)
# Join coverage tables show how well vend and consumption resolve back to Consumer Master.
display(analytics["join_coverage"]["vend"]["coverage"])
display(analytics["join_coverage"]["consumption"]["coverage"])

# This watchlist surfaces consumers with higher exception scores and weaker balance positions.
consumer_summary[
    [
        "consumername",
        "consumernumber",
        "meterno",
        "meterbalance",
        "average_daily_kwh",
        "vend_transaction_count",
        "exception_score",
        "feedercode",
        "dtcode",
    ]
].sort_values(["exception_score", "meterbalance"], ascending=[False, True]).head(20)

## Example analysis: low balance by feeder

In [ ]:
# Group the portfolio to see which feeders have the largest concentration of balance risk.
low_balance_by_feeder = (
    consumer_summary.dropna(subset=["feedercode"])
    .groupby("feedercode", dropna=True)
    .agg(
        consumer_count=("consumer_master_id", "size"),
        low_balance_count=("low_balance_flag", lambda s: int(s.fillna(False).sum())),
        critical_balance_count=("critical_balance_flag", lambda s: int(s.fillna(False).sum())),
        average_daily_kwh=("average_daily_kwh", "mean"),
    )
    .reset_index()
)

low_balance_by_feeder = low_balance_by_feeder.sort_values(
    ["critical_balance_count", "low_balance_count", "consumer_count"],
    ascending=[False, False, False],
)

# Show the raw summary table before plotting so it is easy to inspect exact values.
display(low_balance_by_feeder.head(20))

# A grouped bar chart helps compare critical vs broader low-balance exposure by feeder.
fig = px.bar(
    low_balance_by_feeder.head(15),
    x="feedercode",
    y=["critical_balance_count", "low_balance_count"],
    barmode="group",
    title="Critical and low-balance customers by feeder",
)
fig.update_layout(template="plotly_white")
fig.show()

## Example analysis: daily consumption trend

In [ ]:
# Aggregate the enriched consumption fact table into a portfolio-wide daily time series.
daily_consumption = (
    consumption_enriched.groupby("date", dropna=True)[["kwh_consumption", "kvah_consumption"]]
    .sum(numeric_only=True)
    .reset_index()
    .sort_values("date")
)
# Rolling average smooths daily volatility and makes the overall trend easier to read.
daily_consumption["rolling_7d_kwh"] = daily_consumption["kwh_consumption"].rolling(7, min_periods=1).mean()

display(daily_consumption.tail(10))

# Plot both the raw daily series and the smoothed 7-day trend together.
fig = px.line(
    daily_consumption,
    x="date",
    y=["kwh_consumption", "rolling_7d_kwh"],
    markers=True,
    title="Daily kWh and 7-day rolling average",
)
fig.update_layout(template="plotly_white")
fig.show()

## Example analysis: vend patterns

In [ ]:
# Work on a copy so ad hoc numeric conversions do not mutate the shared enriched frame.
vend_work = vend_enriched.copy()
# Keep vend amounts numeric for aggregation and plotting.
vend_work["transactionamount_numeric"] = pd.to_numeric(vend_work["transactionamount"], errors="coerce")

# Choose the safest analysis path based on issuedate quality.
if vend_timestamp_quality["supports_daily_analysis"]:
    # When calendar dates are reliable, aggregate to a daily vend series.
    daily_vend = (
        vend_work.dropna(subset=["vend_date"])
        .groupby("vend_date", dropna=True)["transactionamount_numeric"]
        .sum()
        .reset_index()
        .sort_values("vend_date")
    )
    display(daily_vend.tail(10))
    fig = px.line(
        daily_vend,
        x="vend_date",
        y="transactionamount_numeric",
        markers=True,
        title="Daily vend amount",
    )
else:
    # If date quality is weak, fall back to hour-of-day patterns instead of faking a calendar trend.
    hourly_vend = (
        vend_work.dropna(subset=["analysis_hour"])
        .groupby("analysis_hour", dropna=True)["transactionamount_numeric"]
        .sum()
        .reset_index()
        .sort_values("analysis_hour")
    )
    display(hourly_vend)
    fig = px.bar(
        hourly_vend,
        x="analysis_hour",
        y="transactionamount_numeric",
        title="Time-of-day vend amount",
    )

# Use the same clean white template as the dashboard charts.
fig.update_layout(template="plotly_white")
fig.show()

## Example analysis: workbench cell

Use this final cell for your own joins, aggregations, plots, or exports.

In [ ]:
# Example starter snippets:
# Use these to discover available fields before writing a custom analysis.
# consumer_summary.columns.tolist()
# vend_enriched.columns.tolist()
# consumption_enriched.columns.tolist()

# your analysis here